# SpecPW-UNet on the FPGA

Cloud / land / sea classification of a HYPSO-2 capture with the quantized SpecPW-UNet: a
spectral stem of 1x1 convs, then a two-level U-Net with one skip connection. A golden
baseline: plain loops in float, int8 weights. The capture is cut into 32 x 32 pixel
patches; the ARM drops 10 bands and z-scores the rest as in training (`mu_sd.txt`), and the
kernel classifies many patches per start, 3 scores per pixel. The highest score is the
class: **0 cloud, 1 land, 2 sea**.

Next to this notebook: the kernel's `.bit` + `.hwh` and its register map `x..._hw.h`:
`sp_unet_small`, `sp_unet_medium` or `sp_unet_large`, the same network in three widths. The
notebook finds which from the files. And `mu_sd.txt`, and an image zip made on the PC:

```bash
python3 tools/justounetsimple_image.py prepare CAPTURE-l1a.nc --labels CAPTURE-l1a_labels.npy --crop 200 300 128 256 --out aegean_unet
python3 tools/specpw_unet_model.py reference aegean_unet
source fpga.sh; fpga_zip aegean_unet
```

The second line writes `reference_sp_unet_SIZE.npy`, what each kernel should answer (the
network in numpy on the PC); without it the classes are checked against the labels only.
Nothing big runs on the ARM. Run the cells from the top.

The image zip can also go in the folder above this one, once for every design: the notebook looks there when it is not next to it, unpacks it there, and writes its own results next to itself.

## 1. Load the bitstream

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

# Which kernel: its register map, x<NAME>_hw.h, lies next to its NAME.bit.
header_file = next(Path('.').glob('x*_hw.h'))
NAME = header_file.name[1:-len('_hw.h')]
overlay = Overlay(f'{NAME}.bit')                 # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design

header = header_file.read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, DIN, DOUT, N = register('AP_CTRL'), register('DIN_DATA'), register('DOUT_DATA'), register('N_DATA')

# The kernel's formats (sp_unet_*.hpp): in, n x 32 x 32 x 110 float32 patches, preprocessed as
# in training: keep 110 of the 120 raw bands, then (x - mean) / std; out, n x 32 x 32 x 3 scores.
P, BANDS, CLASSES = 32, 110, 3
mu, sd = [np.array(a.replace(',', ' ').split(), dtype=np.float64)
          for a in re.findall(r'\[([^\]]*)\]', Path('mu_sd.txt').read_text())]
KEPT = [b for b in range(120) if b not in (0, 1, 2, 3, 4, 5, 6, 7, 118, 119)]
MEAN, INV_STD = mu.astype(np.float32), (1 / sd).astype(np.float32)

def launch(din_address, dout_address, n):
    # Start the kernel on n patches; the addresses are 64 bits: low word, then high word.
    for offset, address in ((DIN, din_address), (DOUT, dout_address)):
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    kernel.write(N, n)
    kernel.write(AP_CTRL, 1)

def wait(timeout):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')
        time.sleep(0.001)

print(f'{NAME}, idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. The functions: cut and preprocess a row of patches, run, a whole image

In [ ]:
def patch_row(cube, y):
    # Image rows y .. y + 31, preprocessed and cut into their 32 x 32 patches (cols x 32 x 32 x 110).
    # The last row and column repeat to fill the edge patches.
    h, w, _ = cube.shape
    rows = np.minimum(np.arange(y, y + P), h - 1)
    cols = np.minimum(np.arange(-(-w // P) * P), w - 1)
    strip = (cube[rows][:, cols][..., KEPT].astype(np.float32) - MEAN) * INV_STD
    return strip.reshape(P, -1, P, BANDS).transpose(1, 0, 2, 3)

def run(n, timeout):
    # n patches from patches_in to scores_out. Returns the FPGA's time, start to done.
    t = time.perf_counter()
    launch(patches_in.physical_address, scores_out.physical_address, n)
    wait(timeout)
    return time.perf_counter() - t

def classify(cube, per_patch):
    # Raw cube (H x W x 120) -> its scores (H x W x 3), and the FPGA's time: ROWS rows of
    # patches per start (all of them when the buffers fit).
    h, w, _ = cube.shape
    scores = np.zeros((ROWS_ALL * P, COLS * P, CLASSES), dtype=np.float32)
    fpga = 0.0
    for first in range(0, ROWS_ALL, ROWS):
        rows = range(first, min(first + ROWS, ROWS_ALL))
        for k, r in enumerate(rows):
            patches_in[k * COLS:(k + 1) * COLS] = patch_row(cube, r * P)
        n = len(rows) * COLS
        patches_in.flush()                     # into DDR, not just the CPU cache
        fpga += run(n, timeout=60 + 3 * per_patch * n)
        scores_out.invalidate()                # read what the kernel wrote
        out = np.asarray(scores_out[:n]).reshape(len(rows), COLS, P, P, CLASSES)
        scores[first * P:(first + len(rows)) * P] = out.transpose(0, 2, 1, 3, 4).reshape(-1, COLS * P, CLASSES)
        print(f'  patch rows {first}-{rows[-1]} done', flush=True)
    return scores[:h, :w], fpga

## 3. Load the image, give the kernel its DDR buffers

In [ ]:
# The image: IMAGE.zip or the folder IMAGE/, next to this notebook or in the folder above
# it. Above is shared by every design's folder: upload the image there once. A zip is
# unpacked next to itself, and again only when it is newer than what was unpacked.
IMAGE = 'aegean_unet'
place = next((d for d in (Path('.'), Path('..'))
              if (d / f'{IMAGE}.zip').exists() or (d / IMAGE / 'cube.npy').exists()), None)
if place is None:
    raise FileNotFoundError(f'no {IMAGE}.zip or {IMAGE}/ in {Path(".").resolve()} or the folder above it')
folder, archive = place / IMAGE, place / f'{IMAGE}.zip'
if archive.exists() and (not (folder / 'cube.npy').exists()
                         or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
    with zipfile.ZipFile(archive) as z:
        z.extractall(place)
print(f'image: {folder.resolve()}')
cube = np.load(folder / 'cube.npy', mmap_mode='r')     # read from the SD card as it is used
meta = json.loads((folder / 'meta.json').read_text())
# What this kernel should answer, made on the PC (specpw_unet_model.py reference), if the image has it.
reference_file = folder / f'reference_{NAME}.npy'
reference = np.load(reference_file) if reference_file.exists() else None
if reference is None:
    print(f'no {reference_file.name} in {folder}: checked against the labels only')
H, W, _ = cube.shape
ROWS_ALL, COLS = -(-H // P), -(-W // P)

# The patches go to DDR the FPGA can read (PYNQ's CMA, shared by every running notebook):
# all of them at once if they fit, else one row of patches per start.
try:
    ROWS = ROWS_ALL
    patches_in = allocate(shape=(ROWS * COLS, P, P, BANDS), dtype=np.float32)
    scores_out = allocate(shape=(ROWS * COLS, P, P, CLASSES), dtype=np.float32)
except Exception as error:                     # not enough (contiguous) CMA
    patches_in = scores_out = None
    ROWS = 1
    patches_in = allocate(shape=(COLS, P, P, BANDS), dtype=np.float32)
    scores_out = allocate(shape=(COLS, P, P, CLASSES), dtype=np.float32)
    print(f'the whole image does not fit in CMA ({error}): one row of patches per start')
print(f'{H} x {W} pixels, {ROWS_ALL} x {COLS} patches, {ROWS * COLS} per start')

## 4. Self-test: the first patch against the PC's answer, and its time

In [ ]:
patches_in[0] = patch_row(cube, 0)[0]
patches_in.flush()
per_patch = run(1, timeout=600)
scores_out.invalidate()
first = np.array(scores_out[0])
print(f'{per_patch:.3f} s per patch on the FPGA: the whole image will take about '
      f'{per_patch * ROWS_ALL * COLS:.0f} s')
if reference is not None:
    want = reference[:P, :P]
    error = np.abs(first[:want.shape[0], :want.shape[1]] - want).max()
    print('PASS' if error < 1e-3 * (1 + np.abs(want).max()) else 'FAIL', f'worst error {error:.2e}')
else:
    print('no reference: the first patch ran, its answers are not checked')

## 5. The whole image

**Wall clock**: from calling `classify` until the ARM has the scores, with the
preprocessing and the patch cutting. **FPGA**: from each start bit until its done bit,
added up.

In [ ]:
start = time.monotonic()
scores, fpga_seconds = classify(cube, per_patch)
wall_seconds = time.monotonic() - start
print(f'wall clock {wall_seconds:.1f} s, FPGA {fpga_seconds:.1f} s for {H} x {W} pixels '
      f'({ROWS_ALL * COLS} patches, {fpga_seconds / (ROWS_ALL * COLS):.3f} s each on the FPGA)')
np.save(f'{IMAGE}_{NAME}_fpga_scores.npy', scores)    # next to this notebook: the image folder is shared

## 6. Check it: against the labels, and the PC's answer if the image has it

In [ ]:
classes, close = scores.argmax(-1), None
if reference is not None:
    close = np.abs(scores - reference) <= 1e-3 + 1e-3 * np.abs(reference)
    print(f'scores within tolerance of the PC: {close.all(-1).mean():.2%}, '
          f'same class: {(classes == reference.argmax(-1)).mean():.2%}')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = np.asarray(cube[:, :, meta['rgb_bands']], dtype=np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          f'{NAME} classes': COLORS[classes]}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
if close is not None:
    panels['differs from PC'] = ~close.all(-1)
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img, cmap='gray_r' if img.ndim == 2 else None)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # cloud white, land green, sea blue, gray = unlabeled